[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/tokenization-embeddings-course/08_tokens_multimodales_audio/08_proyecto_tokens_multimodales_audio.ipynb)

# Proyecto 08 · Tokeniza una foto real y un clip de audio: ¿cuánto comprimes y cuánto pierdes?

| | |
|---|---|
| **Nivel** | 🟠 Avanzado |
| **Duración estimada** | ≈ 2.5 h |
| **GPU recomendada** | `FAST_DEV_RUN=True`: CPU (≈ 4-6 min). Modo completo: T4 (≈ 2-4 unidades, estimación) |
| **Prerrequisitos** | Lección 08 (VQ, RVQ, quantizer dropout, tasa-distorsión, μ-law) |

## Contexto de negocio
En **Políglota Labs** alguien propone guardar las capturas de los PDFs y los audios de soporte como **tokens discretos** «para que un LLM los lea después». Antes de decidir, el CTO quiere números:
¿**cuántos bits** cuesta cada imagen/segundo de audio y **cuánta calidad** se pierde frente a los códecs clásicos (JPEG, μ-law)? Construyes y evalúas los dos tokenizers (imagen y audio) con datos que **no** vio el entrenamiento.

**Conexión.** Lección 08 (VQ-VAE/RVQ/códec toy) → este proyecto; módulo 16: retrieval multimodal con embeddings **continuos** (verás por qué no usar estos tokens para buscar).

## Dataset
* **Imagen:** dos fotos reales empaquetadas con scikit-learn: entrenas con recortes de `china.jpg` y evalúas en `flower.jpg` completa (generalización a otra foto). Respaldo: imagen sintética.
* **Audio:** voz **real** de la lección: *Free Spoken Digit Dataset* (dígitos hablados en inglés, 6 hablantes, 8 kHz, CC BY-SA 4.0) desde su repositorio oficial en GitHub, concatenado en clips de 1 s; el test usa otras grabaciones. Respaldo sin red: voz sintética «tipo voz» (con aviso). LibriSpeech queda como extensión.

## Entregables y criterios de evaluación
| Criterio | Umbral (calibrado con la solución de referencia) |
|---|---|
| R1 · bpp con 1 nivel (fórmula correcta) | = 0.5 (106x160 tokens x 8 bits / 424x640 px) |
| R2 · PSNR en la foto no vista (1 nivel) | ≥ 19.5 dB (medido: 20.45 dB; JPEG al mismo bpp: 32.9 dB) |
| R3 · curva JPEG medida con 7 calidades | PSNR creciente |
| R4 · tasa del códec con 4 niveles | = 4.0 kbps |
| R5 · SNR de audio con 4 niveles y ganancia sobre 1 nivel | ≥ 2.5 dB y ≥ +1.0 dB (medido con voz real FSDD: 2.94 dB con 4 niveles vs 1.46 con 1; con el respaldo sintético, 10.1 vs 5.7) |
| R6 · conclusión escrita en `CONCLUSION` | ≥ 300 caracteres |

In [ ]:
!pip install -q torch torchvision torchaudio numpy matplotlib scikit-learn scipy transformers

In [ ]:
import os, sys, time, math, random, warnings
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore")
import numpy as np
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
FAST_DEV_RUN = True      # True: CPU, minutos. False: GPU (T4/L4), escala completa
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    torch.set_num_threads(int(os.environ.get("OMP_NUM_THREADS", min(4, os.cpu_count() or 1))))
print("device:", device, "| FAST_DEV_RUN:", FAST_DEV_RUN, "| torch", torch.__version__)

import re, io
import matplotlib.pyplot as plt
import torch.nn as nn, torch.nn.functional as F
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .25,
                     "axes.spines.top": False, "axes.spines.right": False})
AZUL, NARANJA, VERDE, GRIS = "#1F6FEB", "#F28E2B", "#2CA02C", "#6B7280"

## Paso 0 · Utilidades y modelos (dados)

Cuantizadores de la lección, la foto, y el entrenamiento de los dos tokenizers (imagen: reducción x4 + RVQ 2x256; audio: códec toy de la lección). Tardan unos minutos en CPU.

In [ ]:
class VQ(nn.Module):
    """Cuantizador vectorial (van den Oord et al., 2017). z: [B,d,*] -> zq (con STE), pérdida, índices [B,*]."""
    def __init__(self, K, d, beta=0.25):
        super().__init__(); self.K, self.d, self.beta = K, d, beta
        self.cb = nn.Embedding(K, d); self.cb.weight.data.uniform_(-1 / K, 1 / K)
        self.register_buffer("inicializado", torch.tensor(False)); self.register_buffer("uso", torch.ones(K))
    def forward(self, z, reinicia=False):
        shape = z.shape; zf = z.movedim(1, -1).reshape(-1, self.d)
        if self.training and not bool(self.inicializado):                       # init por datos: codebook = vectores del encoder
            self.cb.weight.data.copy_(zf[torch.randperm(len(zf), device=z.device)[:self.K]].detach()); self.inicializado.fill_(True)
        dist = zf.pow(2).sum(1, keepdim=True) - 2 * zf @ self.cb.weight.T + self.cb.weight.pow(2).sum(1)
        idx = dist.argmin(1); e = self.cb(idx)
        loss = F.mse_loss(e, zf.detach()) + self.beta * F.mse_loss(zf, e.detach())     # ||sg[z]-e||² + β||z-sg[e]||²
        if self.training:
            self.uso.mul_(0.99).add_(torch.bincount(idx, minlength=self.K).float(), alpha=0.01 * 1.0)
            if reinicia:                                                                  # reinicio de códigos muertos
                muertos = self.uso < 1e-3 * len(idx) / self.K
                if muertos.any(): self.cb.weight.data[muertos] = zf[torch.randint(len(zf), (int(muertos.sum()),), device=z.device)].detach()
        zq = zf + (e - zf).detach()                                                       # straight-through estimator
        return zq.reshape(*shape[:1], *shape[2:], self.d).movedim(-1, 1), loss, idx.reshape(shape[0], *shape[2:])

class FSQ(nn.Module):
    """Finite Scalar Quantization (Mentzer et al., 2023): acota cada canal y lo redondea a L_i niveles; sin codebook ni pérdidas extra.
    (Normalizamos z por su desviación típica por canal: con salidas del encoder ~0.1 todo redondearía a 0 y el entrenamiento colapsa.)"""
    def __init__(self, levels):
        super().__init__(); L = torch.tensor(levels); self.register_buffer("L", L); self.d = len(levels)
        self.register_buffer("basis", torch.cumprod(torch.tensor([1] + list(levels[:-1])), 0)); self.K = int(L.prod())
    def forward(self, z, reinicia=False):
        zp = z.movedim(1, -1); zp = zp / (zp.detach().reshape(-1, self.d).std(0) + 1e-5); half = (self.L - 1) * (1 - 1e-3) / 2; off = torch.where(self.L % 2 == 0, 0.5, 0.0); shift = torch.atanh(off / half)
        b = torch.tanh(zp + shift) * half - off; q = b + (torch.round(b) - b).detach()          # acota + redondea (STE)
        hw = (self.L // 2).float(); idx = ((q + hw) * self.basis).sum(-1).long()
        return (q / hw).movedim(-1, 1), z.new_zeros(()), idx

class RVQ(nn.Module):
    """Cuantización vectorial residual: cada nivel cuantiza lo que el anterior dejó sin explicar."""
    def __init__(self, Q, K, d, beta=0.25):
        super().__init__(); self.Q, self.K = Q, K; self.capas = nn.ModuleList([VQ(K, d, beta) for _ in range(Q)])
    def forward(self, z, reinicia=False, n_q=None):
        res, out, loss, idxs = z, 0, 0, []
        for capa in self.capas[: (n_q or self.Q)]:
            q, l, idx = capa(res, reinicia); res = res - q.detach(); out = out + q; loss = loss + l; idxs.append(idx)
        return out, loss, torch.stack(idxs, 1)

def perplexity(idx, K):
    p = torch.bincount(idx.flatten(), minlength=K).float(); p = p / p.sum()
    return float(torch.exp(-(p[p > 0] * p[p > 0].log()).sum()))

def psnr(x, y, peak=1.0):
    mse = F.mse_loss(x, y).item(); return 10 * math.log10(peak ** 2 / max(mse, 1e-10))

In [ ]:
from PIL import Image, ImageDraw

def mnist_sintetico(n, seed=SEED):
    """Respaldo sin red: figuras geométricas 28x28 (círculo, cuadrado, cruz, triángulo) con posición/tamaño aleatorios."""
    r = np.random.RandomState(seed); X = np.zeros((n, 1, 28, 28), np.float32); y = r.randint(0, 4, n)
    for i in range(n):
        im = Image.new("L", (28, 28)); d = ImageDraw.Draw(im); c = r.randint(9, 19, 2); s = r.randint(4, 9)
        box = [c[0] - s, c[1] - s, c[0] + s, c[1] + s]
        if y[i] == 0: d.ellipse(box, outline=255, width=2)
        elif y[i] == 1: d.rectangle(box, outline=255, width=2)
        elif y[i] == 2: d.line([c[0] - s, c[1], c[0] + s, c[1]], fill=255, width=2); d.line([c[0], c[1] - s, c[0], c[1] + s], fill=255, width=2)
        else: d.polygon([(c[0], c[1] - s), (c[0] - s, c[1] + s), (c[0] + s, c[1] + s)], outline=255)
        X[i, 0] = np.asarray(im, np.float32) / 255
    return torch.from_numpy(X), torch.from_numpy(y)

def carga_imagenes(n):
    try:
        from torchvision import datasets
        ds = datasets.MNIST("./data", train=True, download=True)
        return ds.data[:n].float().div(255).unsqueeze(1), ds.targets[:n], "MNIST real"
    except Exception as e:
        print("[aviso] MNIST no disponible (", type(e).__name__, ") -> figuras SINTÉTICAS 28x28")
        X, y = mnist_sintetico(n); return X, y, "sintético (4 clases)"

def foto(nombre="china.jpg"):
    """Foto real que viene EMPAQUETADA con scikit-learn (sin descargas); respaldo sintético si no está."""
    try:
        from sklearn.datasets import load_sample_image
        return Image.fromarray(load_sample_image(nombre)), "foto real (scikit-learn)"
    except Exception:
        r = np.random.RandomState(0); yy, xx = np.mgrid[0:427, 0:640]
        im = np.stack([128 + 100 * np.sin(xx / 40 + k) * np.cos(yy / (30 + 10 * k)) + r.randn(427, 640) * 6 for k in range(3)], -1)
        return Image.fromarray(np.clip(im, 0, 255).astype(np.uint8)), "imagen sintética"

In [ ]:
im_tr, fuente_tr = foto("china.jpg"); im_te, fuente_te = foto("flower.jpg")
A_TRAIN = torch.from_numpy(np.asarray(im_tr, np.float32) / 255).permute(2, 0, 1).to(device)          # [3,427,640]
TEST_IMG = torch.from_numpy(np.asarray(im_te, np.float32) / 255).permute(2, 0, 1)[:, :424, :640][None].to(device)
print(f"entrenamiento: {fuente_tr} china.jpg {tuple(A_TRAIN.shape)} | test (otra foto): flower.jpg {tuple(TEST_IMG.shape)}")

def recortes(B, s=64, g=None):
    ys = torch.randint(0, A_TRAIN.shape[1] - s, (B,), generator=g); xs = torch.randint(0, A_TRAIN.shape[2] - s, (B,), generator=g)
    return torch.stack([A_TRAIN[:, y:y + s, x:x + s] for y, x in zip(ys.tolist(), xs.tolist())])

class AEFoto(nn.Module):
    """Autoencoder conv RGB con reducción x4 por eje (una rejilla de tokens de H/4 x W/4) y RVQ de 2 niveles de 256 códigos."""
    def __init__(self, d=32, Q=2, K=256):
        super().__init__()
        self.enc = nn.Sequential(nn.Conv2d(3, 64, 4, 2, 1), nn.ReLU(), nn.Conv2d(64, 128, 4, 2, 1), nn.ReLU(), nn.Conv2d(128, d, 1))
        self.quant = RVQ(Q, K, d)
        self.dec = nn.Sequential(nn.Conv2d(d, 128, 3, 1, 1), nn.ReLU(), nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.ReLU(),
                                 nn.ConvTranspose2d(64, 3, 4, 2, 1), nn.Sigmoid())
    def forward(self, x, reinicia=False, n_q=None):
        zq, ql, idx = self.quant(self.enc(x), reinicia, n_q); return self.dec(zq), ql, idx

torch.manual_seed(SEED); AE_F = AEFoto().to(device); opt = torch.optim.Adam(AE_F.parameters(), lr=2e-3); g = torch.Generator().manual_seed(SEED)
PASOS_F = 500 if FAST_DEV_RUN else 5000; t0 = time.time()
for s in range(PASOS_F):
    AE_F.train(); x = recortes(16, g=g); nq = int(torch.randint(1, 3, (1,), generator=g))           # quantizer dropout (1 o 2 niveles)
    xr, ql, _ = AE_F(x, reinicia=(s % 50 == 0 and s > 0), n_q=nq); loss = F.mse_loss(xr, x) + ql; opt.zero_grad(); loss.backward(); opt.step()
print(f"tokenizer de fotos entrenado en {time.time()-t0:.0f}s | pérdida final {loss.item():.4f}")

In [ ]:
from scipy.signal import lfilter
SR = 8000; DUR = 1.0; N_AUD = int(SR * DUR)
VOCALES = {"a": (730, 1090, 2440), "e": (530, 1840, 2480), "i": (270, 2290, 3010), "o": (570, 840, 2410), "u": (300, 870, 2240)}

def resonador(x, f, bw=80):
    """Filtro de 2º orden (formante) a la frecuencia f con ancho de banda bw."""
    r = np.exp(-np.pi * bw / SR); th = 2 * np.pi * f / SR
    return lfilter([1 - r], [1, -2 * r * np.cos(th), r * r], x)

def voz_sintetica(rng, dur=DUR):
    """Clip 'tipo voz': tren de pulsos glotales con f0 variable + formantes de vocales + envolvente silábica + ruido."""
    n = int(SR * dur); t = np.arange(n) / SR
    f0 = rng.uniform(100, 220) * (1 + 0.05 * np.sin(2 * np.pi * rng.uniform(3, 6) * t) + 0.15 * (t / dur - .5))
    fase = np.cumsum(f0) / SR; src = np.sign(np.diff(np.floor(fase), prepend=0)) * 1.0           # pulsos
    src = lfilter([1], [1, -0.97], src)                                                          # espectro -6 dB/oct
    y = np.zeros(n); seg = int(SR * rng.uniform(0.18, 0.3)); i = 0
    while i < n:
        F1, F2, F3 = VOCALES[rng.choice(list(VOCALES))]; j = min(n, i + seg)
        s = src[i:j]; v = resonador(s, F1) + 0.6 * resonador(s, F2, 100) + 0.3 * resonador(s, F3, 120)
        env = np.hanning(len(v)) ** 0.5; y[i:j] += v * env; i = j
    y += rng.randn(n) * 0.01 * np.abs(y).max(); y = y / (np.abs(y).max() + 1e-8) * 0.8
    return y.astype(np.float32)

def carga_fsdd(n_tr, n_te, seed=SEED):
    """Voz REAL: Free Spoken Digit Dataset (6 hablantes, dígitos en inglés, 8 kHz mono, CC BY-SA 4.0), desde su repositorio
    oficial en GitHub. Las grabaciones (~0.4 s) se concatenan y se cortan en clips de 1 s; test = otras grabaciones (índices 45-49)."""
    import io, urllib.request, concurrent.futures as cf
    from scipy.io import wavfile
    base = "https://raw.githubusercontent.com/Jakobovski/free-spoken-digit-dataset/master/recordings/"
    spk = ["george", "jackson", "lucas", "nicolas", "theo", "yweweler"]
    lista = lambda idx: [f"{d}_{s}_{i}.wav" for d in range(10) for s in spk for i in idx]
    def baja(nombre):
        try:
            with urllib.request.urlopen(base + nombre, timeout=20) as r:
                sr, x = wavfile.read(io.BytesIO(r.read()))
        except Exception:
            return None
        x = x.astype(np.float32)
        return x / (np.abs(x).max() + 1e-8) * 0.8 if sr == SR and x.ndim == 1 else None
    def clips(nombres, n):
        with cf.ThreadPoolExecutor(16) as ex:
            xs = [x for x in ex.map(baja, nombres) if x is not None]
        sig = np.concatenate(xs); k = min(n, len(sig) // N_AUD)
        return sig[: k * N_AUD].reshape(k, N_AUD)
    r = np.random.RandomState(seed + 1)                                     # rng propio: no altera el respaldo sintético
    tr, te = list(r.permutation(lista(range(45)))), list(r.permutation(lista(range(45, 50))))
    return clips(tr[: int(2.6 * n_tr) + 10], n_tr), clips(te[: int(2.6 * n_te) + 10], n_te)

rng = np.random.RandomState(SEED)
N_TR_A = 192 if FAST_DEV_RUN else 2000                                       # sin FAST, FSDD da ~1 200 s de train (se usan todos)
try:
    a_tr, a_te = carga_fsdd(N_TR_A, 24)
    assert len(a_tr) >= min(0.9 * N_TR_A, 500) and len(a_te) >= 12, "descarga incompleta"
    A_TR = torch.from_numpy(a_tr)[:, None].to(device); A_TE = torch.from_numpy(a_te)[:, None].to(device)
    FUENTE_AUD = "voz REAL (Free Spoken Digit Dataset, GitHub)"
except Exception as e:
    print("⚠️  [aviso] FSDD no disponible (", type(e).__name__, ") -> voz SINTÉTICA 'tipo voz' (mismo formato; cifras NO comparables con voz real)")
    A_TR = torch.from_numpy(np.stack([voz_sintetica(rng) for _ in range(N_TR_A)]))[:, None].to(device)
    A_TE = torch.from_numpy(np.stack([voz_sintetica(rng) for _ in range(24)]))[:, None].to(device)
    FUENTE_AUD = "audio sintético 'tipo voz'"
print(f"{FUENTE_AUD}: train={tuple(A_TR.shape)} test={tuple(A_TE.shape)} a {SR} Hz  (PCM 16 bit = {SR*16/1000:.0f} kbps)")

In [ ]:
class CodecToy(nn.Module):
    """Códec neuronal de juguete (SoundStream/EnCodec en miniatura): encoder conv (stride total 64) -> RVQ -> decoder conv."""
    def __init__(self, d=32, Q=4, K=256):
        super().__init__()
        self.enc = nn.Sequential(nn.Conv1d(1, 16, 7, 1, 3), nn.ELU(), nn.Conv1d(16, 32, 4, 2, 1), nn.ELU(), nn.Conv1d(32, 64, 8, 4, 2), nn.ELU(),
                                 nn.Conv1d(64, d, 16, 8, 4))
        self.rvq = RVQ(Q, K, d)
        self.dec = nn.Sequential(nn.ConvTranspose1d(d, 64, 16, 8, 4), nn.ELU(), nn.ConvTranspose1d(64, 32, 8, 4, 2), nn.ELU(),
                                 nn.ConvTranspose1d(32, 16, 4, 2, 1), nn.ELU(), nn.Conv1d(16, 1, 7, 1, 3))
    def forward(self, x, reinicia=False, n_q=None):
        zq, ql, idx = self.rvq(self.enc(x), reinicia, n_q); return self.dec(zq), ql, idx

def stft_mag(x, n_fft):
    return torch.stft(x.squeeze(1), n_fft, n_fft // 4, window=torch.hann_window(n_fft, device=x.device), return_complex=True).abs()

def perdida_audio(x, xr):
    """L1 en la forma de onda + L1 de log-magnitud STFT multi-escala (la idea de EnCodec/DAC; sin discriminador)."""
    l = 10 * F.l1_loss(xr, x)                                   # peso 10: sin él el códec ignora la fase (SNR < 0 dB)
    for n in (512, 256, 128): l = l + 0.5 * F.l1_loss(torch.log(stft_mag(xr, n) + 1e-5), torch.log(stft_mag(x, n) + 1e-5))
    return l

def snr_db(x, xr): return float(10 * torch.log10(x.pow(2).sum() / (x - xr).pow(2).sum().clamp_min(1e-10)))
def lsd_db(x, xr, n=512):
    a, b = 20 * torch.log10(stft_mag(x, n) + 1e-5), 20 * torch.log10(stft_mag(xr, n) + 1e-5); return float((a - b).pow(2).mean(-2).sqrt().mean())

Q_A, K_A, FRAMES = 4, 256, N_AUD // 64
torch.manual_seed(SEED); cod = CodecToy(Q=Q_A, K=K_A).to(device); opt = torch.optim.Adam(cod.parameters(), lr=1e-3)
PASOS_A = 400 if FAST_DEV_RUN else 4000; g = torch.Generator().manual_seed(SEED); t0 = time.time(); hist_a = []
for s in range(PASOS_A):
    cod.train(); x = A_TR[torch.randint(0, len(A_TR), (16,), generator=g).to(device)]; nq = int(torch.randint(1, Q_A + 1, (1,), generator=g))   # quantizer dropout
    xr, ql, _ = cod(x, reinicia=(s % 50 == 0 and s > 0), n_q=nq); loss = perdida_audio(x, xr) + ql; opt.zero_grad(); loss.backward(); opt.step()
    if s % 25 == 0: hist_a.append((s, loss.item()))
print(f"códec toy: {sum(p.numel() for p in cod.parameters())/1e6:.2f}M params | {time.time()-t0:.0f}s | {FRAMES} frames/s = {SR//64} Hz | pérdida {hist_a[0][1]:.2f} -> {hist_a[-1][1]:.2f}")

In [ ]:
def mulaw_snr(x, bits, mu=255.0):
    y = torch.sign(x) * torch.log1p(mu * x.abs()) / math.log1p(mu); q = torch.round((y + 1) / 2 * (2 ** bits - 1)) / (2 ** bits - 1) * 2 - 1
    return snr_db(x, torch.sign(q) * ((1 + mu) ** q.abs() - 1) / mu)

## Paso 1 · TODO 1-2: medir la imagen

$\text{bpp} = \dfrac{N_{\text{tokens}}\cdot n_q\cdot\log_2 K}{H\cdot W}$. Para JPEG, mide los **bytes reales** del archivo.

In [ ]:
def bpp(n_tokens, K, H, W, n_q=1):
    """Bits por píxel de una imagen H x W representada con n_tokens posiciones, n_q niveles de un codebook de K entradas."""
    # TODO 1
    raise NotImplementedError("TODO 1")

In [ ]:
def jpeg_punto(img, calidad):
    """img: tensor [1,3,H,W] en [0,1]. Comprime con JPEG (PIL) a la calidad dada y devuelve (bpp_real, psnr_dB).
    bpp_real = 8 * nº de bytes del archivo JPEG / (H*W)."""
    # TODO 2: pasa a uint8 HxWx3, guarda en io.BytesIO con Image.save(..., format="JPEG", quality=calidad), mide bytes, decodifica y calcula psnr
    raise NotImplementedError("TODO 2")

<details>
<summary>🪜 Pista 1</summary>

`io.BytesIO()` como archivo en memoria; tras `save`, `buf.tell()` da el nº de bytes. Vuelve al inicio con `buf.seek(0)` y abre con `Image.open(buf)`.

</details>

<details>
<summary>🪜 Pista 2</summary>

`arr = (img[0].permute(1,2,0).clamp(0,1).cpu().numpy()*255).round().astype(np.uint8)`; reconstrucción: `torch.from_numpy(np.asarray(Image.open(buf), np.float32)/255).permute(2,0,1)[None]`; usa `psnr(rec, img.cpu())`.

</details>

In [ ]:
RES = {}
try:
    AE_F.eval(); H_, W_ = TEST_IMG.shape[2:]
    for nq in [1, 2]:
        with torch.no_grad(): xr, _, idx = AE_F(TEST_IMG, n_q=nq)
        RES[f"img_{nq}"] = {"bpp": bpp(idx.shape[2] * idx.shape[3], 256, H_, W_, nq), "psnr": psnr(xr, TEST_IMG), "xr": xr.cpu()}
    RES["jpeg"] = [jpeg_punto(TEST_IMG, q) for q in [5, 10, 20, 35, 50, 75, 90]]
    for nq in [1, 2]: print(f"VQ-RVQ {nq} nivel(es): {RES[f'img_{nq}']['bpp']:.3f} bpp | PSNR {RES[f'img_{nq}']['psnr']:.2f} dB")
    print("JPEG (bpp, PSNR):", [(round(b, 3), round(p, 2)) for b, p in RES["jpeg"]])
    jb = np.array([b for b, _ in RES["jpeg"]]); jp = np.array([p for _, p in RES["jpeg"]]); o = np.argsort(jb)
    RES["jpeg_a_mismo_bpp"] = float(np.interp(RES["img_1"]["bpp"], jb[o], jp[o]))
    print(f"JPEG interpolado a {RES['img_1']['bpp']:.2f} bpp: {RES['jpeg_a_mismo_bpp']:.2f} dB")
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 2 · TODO 3-4: tokenizar el audio con los codebooks entrenados

Reimplementa la **codificación RVQ** usando solo las matrices `capa.cb.weight` (esto es lo que haría un servidor que guarda los ids) y calcula la tasa.

In [ ]:
def rvq_codifica(z, capas, n_q):
    """Cuantización residual SOLO con los codebooks (sin módulos): z [B,d,T]; capas = lista de VQ ya entrenados.
    Devuelve (ids [B, n_q, T], z_hat [B,d,T]). Para cada nivel: vecino más cercano del residuo en capa.cb.weight."""
    # TODO 3: r = z; para cada capa en capas[:n_q]: ids = argmin distancia(r, codebook); q = codebook[ids]; r = r - q; acumula q
    raise NotImplementedError("TODO 3")

<details>
<summary>🪜 Pista 1</summary>

Trabaja con `r = z.movedim(1, -1)` ([B,T,d]) para que la distancia a la matriz `cb` [K,d] sea `r.pow(2).sum(-1,keepdim=True) - 2*r@cb.T + cb.pow(2).sum(1)`.

</details>

<details>
<summary>🪜 Pista 2</summary>

Por nivel: `i = d.argmin(-1); q = cb[i]; r = r - q; zh = zh + q; ids.append(i)`. Al final `torch.stack(ids, 1)` y `zh.movedim(-1, 1)`.

</details>

In [ ]:
def kbps_codec(sr, stride, n_q, K):
    """Tasa de un códec RVQ en kbps: (sr/stride) frames/s x n_q niveles x log2(K) bits."""
    # TODO 4
    raise NotImplementedError("TODO 4")

In [ ]:
try:
    cod.eval(); RES["aud"] = {}
    with torch.no_grad():
        z = cod.enc(A_TE)
        for nq in range(1, Q_A + 1):
            ids, zh = rvq_codifica(z, list(cod.rvq.capas), nq); xr = cod.dec(zh)
            RES["aud"][nq] = {"kbps": kbps_codec(SR, 64, nq, K_A), "snr": snr_db(A_TE, xr), "lsd": lsd_db(A_TE, xr), "ids": tuple(ids.shape)}
    for nq, r in RES["aud"].items(): print(f"audio n_q={nq}: {r['kbps']:.1f} kbps | SNR {r['snr']:.2f} dB | LSD {r['lsd']:.2f} dB | ids {r['ids']}")
    RES["mulaw"] = {b: (mulaw_snr(A_TE, b), SR * b / 1000) for b in [3, 4, 5, 6, 8]}
except (NotImplementedError, NameError) as e:
    print("Pendiente:", e)

In [ ]:
if "img_1" in RES and "aud" in RES:
    fig, axs = plt.subplots(1, 4, figsize=(17, 3.8))
    axs[0].imshow(TEST_IMG[0].permute(1, 2, 0).cpu()); axs[0].set_title("original (flower.jpg, no vista)", fontsize=9)
    axs[1].imshow(RES["img_1"]["xr"][0].permute(1, 2, 0).clamp(0, 1)); axs[1].set_title(f"tokens VQ, {RES['img_1']['bpp']:.2f} bpp, {RES['img_1']['psnr']:.1f} dB", fontsize=9)
    for a in axs[:2]: a.axis("off"); a.grid(False)
    axs[2].plot([b for b, _ in RES["jpeg"]], [p for _, p in RES["jpeg"]], "s--", color=NARANJA, label="JPEG")
    axs[2].plot([RES[f"img_{q}"]["bpp"] for q in [1, 2]], [RES[f"img_{q}"]["psnr"] for q in [1, 2]], "o-", color=AZUL, lw=2, label="VQ-RVQ (1-2 niveles)")
    axs[2].set_xlabel("bits por píxel"); axs[2].set_ylabel("PSNR (dB)"); axs[2].set_title("Imagen: tasa-distorsión"); axs[2].legend(fontsize=8)
    axs[3].plot([r["kbps"] for r in RES["aud"].values()], [r["snr"] for r in RES["aud"].values()], "o-", color=AZUL, lw=2, label="códec RVQ")
    axs[3].plot([v[1] for v in RES["mulaw"].values()], [v[0] for v in RES["mulaw"].values()], "s--", color=NARANJA, label="μ-law")
    axs[3].set_xscale("log"); axs[3].set_xlabel("kbps"); axs[3].set_ylabel("SNR (dB)"); axs[3].set_title("Audio: tasa-fidelidad"); axs[3].legend(fontsize=8)
    plt.tight_layout(); plt.show()
else:
    print("Completa los TODO 1-4 para ver la figura.")

> **👀 Qué debes observar**
>
> **Paneles 1–2:** la reconstrucción con tokens VQ pierde detalle fino (bordes, textura) pero conserva la estructura; su título da los bpp y el PSNR. **Panel 3 (imagen):** compara las dos curvas a igual bpp; si JPEG queda por encima, no es un fallo tuyo: un códec diseñado durante décadas para el ojo puede comprimir mejor que un tokenizer diminuto entrenado en minutos. **Panel 4 (audio):** cada nivel RVQ extra sube los kbps y el SNR; μ-law es la referencia clásica. La conclusión que debes poder defender: estos tokens no se diseñan para ganar a JPEG o μ-law, sino para que un **transformer** pueda modelar imágenes y audio como secuencias discretas (lección, §8).

## Paso 3 · Conclusión y rúbrica

Escribe en `CONCLUSION` (≥ 300 caracteres): ¿le ganan tus tokens a JPEG/μ-law? ¿para qué sirven entonces? ¿qué recomendarías a Políglota y qué **no** puedes concluir?

In [ ]:
CONCLUSION = ""   # TODO 5

In [ ]:
THR = {'bpp_1': 0.5, 'psnr_img_min': 19.5, 'jpeg_ventaja_min': 0.0, 'kbps_4': 4.0, 'snr_4_min': 2.5, 'ganancia_snr_min': 1.0}

def rubrica(RES, th=THR):
    filas = []; chk = lambda n, ok, d: filas.append((n, ok, d))
    i1 = RES.get("img_1", {}); a = RES.get("aud", {})
    chk("R1 bpp de la imagen con 1 nivel = %.2f (tu fórmula)" % th["bpp_1"], abs(i1.get("bpp", -1) - th["bpp_1"]) < 1e-6, f"{i1.get('bpp')}")
    chk("R2 PSNR en la foto NO vista con 1 nivel >= %.1f dB" % th["psnr_img_min"], i1.get("psnr", 0) >= th["psnr_img_min"], f"{i1.get('psnr')}")
    chk("R3 tu curva JPEG está bien medida (PSNR crece con la calidad)", len(RES.get("jpeg", [])) == 7 and RES["jpeg"][-1][1] > RES["jpeg"][0][1], f"{RES.get('jpeg')}")
    chk("R4 tasa del códec con 4 niveles = %.1f kbps" % th["kbps_4"], abs(a.get(4, {}).get("kbps", -1) - th["kbps_4"]) < 1e-6, f"{a.get(4, {}).get('kbps')}")
    chk("R5 SNR con 4 niveles >= %.1f dB y mejora >= %.1f dB sobre 1 nivel" % (th["snr_4_min"], th["ganancia_snr_min"]),
        4 in a and a[4]["snr"] >= th["snr_4_min"] and a[4]["snr"] - a[1]["snr"] >= th["ganancia_snr_min"], f"{[round(a[q]['snr'], 2) for q in a] if a else None}")
    chk("R6 conclusión escrita (>= 300 caracteres) en CONCLUSION", len(globals().get("CONCLUSION", "")) >= 300, f"{len(globals().get('CONCLUSION', ''))} caracteres")
    for n, ok, d in filas: print(("✅" if ok else "❌"), n, "|", d)
    print(f"\n{sum(o for _, o, _ in filas)}/{len(filas)} criterios")

rubrica(RES)

---
## ⛔ SPOILER — intenta resolverlo primero

Solución de referencia completa y ejecutable.

In [ ]:
def bpp(n_tokens, K, H, W, n_q=1):
    return n_tokens * n_q * math.log2(K) / (H * W)

In [ ]:
def jpeg_punto(img, calidad):
    arr = (img[0].permute(1, 2, 0).clamp(0, 1).cpu().numpy() * 255).round().astype(np.uint8)
    buf = io.BytesIO(); Image.fromarray(arr).save(buf, format="JPEG", quality=calidad); n = buf.tell(); buf.seek(0)
    rec = torch.from_numpy(np.asarray(Image.open(buf), np.float32) / 255).permute(2, 0, 1)[None]
    return 8 * n / (arr.shape[0] * arr.shape[1]), psnr(rec, img.cpu())

In [ ]:
def rvq_codifica(z, capas, n_q):
    r = z.movedim(1, -1); zh = torch.zeros_like(r); ids = []
    for capa in capas[:n_q]:
        cb = capa.cb.weight; d = r.pow(2).sum(-1, keepdim=True) - 2 * r @ cb.T + cb.pow(2).sum(1)
        i = d.argmin(-1); q = cb[i]; r = r - q; zh = zh + q; ids.append(i)
    return torch.stack(ids, 1), zh.movedim(-1, 1)

In [ ]:
def kbps_codec(sr, stride, n_q, K):
    return sr / stride * n_q * math.log2(K) / 1000

In [ ]:
RES = {}
try:
    AE_F.eval(); H_, W_ = TEST_IMG.shape[2:]
    for nq in [1, 2]:
        with torch.no_grad(): xr, _, idx = AE_F(TEST_IMG, n_q=nq)
        RES[f"img_{nq}"] = {"bpp": bpp(idx.shape[2] * idx.shape[3], 256, H_, W_, nq), "psnr": psnr(xr, TEST_IMG), "xr": xr.cpu()}
    RES["jpeg"] = [jpeg_punto(TEST_IMG, q) for q in [5, 10, 20, 35, 50, 75, 90]]
    for nq in [1, 2]: print(f"VQ-RVQ {nq} nivel(es): {RES[f'img_{nq}']['bpp']:.3f} bpp | PSNR {RES[f'img_{nq}']['psnr']:.2f} dB")
    print("JPEG (bpp, PSNR):", [(round(b, 3), round(p, 2)) for b, p in RES["jpeg"]])
    jb = np.array([b for b, _ in RES["jpeg"]]); jp = np.array([p for _, p in RES["jpeg"]]); o = np.argsort(jb)
    RES["jpeg_a_mismo_bpp"] = float(np.interp(RES["img_1"]["bpp"], jb[o], jp[o]))
    print(f"JPEG interpolado a {RES['img_1']['bpp']:.2f} bpp: {RES['jpeg_a_mismo_bpp']:.2f} dB")
except NotImplementedError as e:
    print("Pendiente:", e)


try:
    cod.eval(); RES["aud"] = {}
    with torch.no_grad():
        z = cod.enc(A_TE)
        for nq in range(1, Q_A + 1):
            ids, zh = rvq_codifica(z, list(cod.rvq.capas), nq); xr = cod.dec(zh)
            RES["aud"][nq] = {"kbps": kbps_codec(SR, 64, nq, K_A), "snr": snr_db(A_TE, xr), "lsd": lsd_db(A_TE, xr), "ids": tuple(ids.shape)}
    for nq, r in RES["aud"].items(): print(f"audio n_q={nq}: {r['kbps']:.1f} kbps | SNR {r['snr']:.2f} dB | LSD {r['lsd']:.2f} dB | ids {r['ids']}")
    RES["mulaw"] = {b: (mulaw_snr(A_TE, b), SR * b / 1000) for b in [3, 4, 5, 6, 8]}
except (NotImplementedError, NameError) as e:
    print("Pendiente:", e)

In [ ]:
if "img_1" in RES and "aud" in RES:
    fig, axs = plt.subplots(1, 4, figsize=(17, 3.8))
    axs[0].imshow(TEST_IMG[0].permute(1, 2, 0).cpu()); axs[0].set_title("original (flower.jpg, no vista)", fontsize=9)
    axs[1].imshow(RES["img_1"]["xr"][0].permute(1, 2, 0).clamp(0, 1)); axs[1].set_title(f"tokens VQ, {RES['img_1']['bpp']:.2f} bpp, {RES['img_1']['psnr']:.1f} dB", fontsize=9)
    for a in axs[:2]: a.axis("off"); a.grid(False)
    axs[2].plot([b for b, _ in RES["jpeg"]], [p for _, p in RES["jpeg"]], "s--", color=NARANJA, label="JPEG")
    axs[2].plot([RES[f"img_{q}"]["bpp"] for q in [1, 2]], [RES[f"img_{q}"]["psnr"] for q in [1, 2]], "o-", color=AZUL, lw=2, label="VQ-RVQ (1-2 niveles)")
    axs[2].set_xlabel("bits por píxel"); axs[2].set_ylabel("PSNR (dB)"); axs[2].set_title("Imagen: tasa-distorsión"); axs[2].legend(fontsize=8)
    axs[3].plot([r["kbps"] for r in RES["aud"].values()], [r["snr"] for r in RES["aud"].values()], "o-", color=AZUL, lw=2, label="códec RVQ")
    axs[3].plot([v[1] for v in RES["mulaw"].values()], [v[0] for v in RES["mulaw"].values()], "s--", color=NARANJA, label="μ-law")
    axs[3].set_xscale("log"); axs[3].set_xlabel("kbps"); axs[3].set_ylabel("SNR (dB)"); axs[3].set_title("Audio: tasa-fidelidad"); axs[3].legend(fontsize=8)
    plt.tight_layout(); plt.show()
else:
    print("Completa los TODO 1-4 para ver la figura.")

In [ ]:
CONCLUSION = (
    f"Imagen: con 1 nivel ({RES['img_1']['bpp']:.2f} bpp) nuestro tokenizer da {RES['img_1']['psnr']:.1f} dB en una foto no vista, frente a ~{RES['jpeg_a_mismo_bpp']:.1f} dB de JPEG al mismo bpp: "
    "un tokenizer diminuto entrenado con una sola foto NO compite con JPEG en fidelidad. "
    f"Audio: el códec toy con 4 niveles ({RES['aud'][4]['kbps']:.0f} kbps, 32x menos que PCM-16) logra {RES['aud'][4]['snr']:.1f} dB de SNR frente a {RES['aud'][1]['snr']:.1f} dB con 1 nivel; μ-law necesita decenas de kbps para SNR parecidos. "
    "El valor de los tokens discretos no es la compresión sino que un LM puede predecirlos y mezclarlos con texto. Recomendación para Políglota: para BUSCAR, embeddings continuos (módulo 16); "
    "tokens discretos solo si hay generación. Límites: una semilla, modelos diminutos sin pérdidas perceptuales/GAN, pocos segundos de voz (dígitos en inglés); PSNR/SNR no son calidad perceptual."
)
rubrica(RES)

## Retos extra (nivel experto)
1. Añade una pérdida **perceptual** sencilla (diferencias de gradiente o LPIPS si tienes acceso) y mide si el PSNR baja pero la imagen *se ve* mejor.
2. Sustituye el RVQ de la imagen por **FSQ** con niveles [8,6,5] y compara uso del codebook y PSNR.
3. Entrena con **más fotos** (p. ej. un subconjunto de COCO si tienes red) y mide la generalización.
4. Prueba **LibriSpeech** real (`openslr/librispeech_asr`) o EnCodec real (`facebook/encodec_24khz`) y compara su SNR/LSD con tu códec.
5. Calcula cuántos **tokens por minuto** generaría tu códec y cuánto ocuparían 1 000 horas de llamadas en almacenamiento y en contexto.

## Reflexión
* ¿Por qué el paper de Chameleon acepta perder calidad al discretizar imágenes?
* ¿Qué métrica perceptual usarías para decidir un códec de audio para Políglota?
* ¿Qué parte de este análisis cambia si el objetivo es *retrieval* (módulo 16) y no generación?